# Fine-tuning QLoRA avec Unsloth : extraction JSON de factures

Ce notebook entraîne un modèle open-weights (par défaut Qwen2.5-3B-Instruct) en QLoRA sur `data/train.jsonl` et `data/val.jsonl`, au format ChatML.

**Prérequis**
- Runtime GPU (Google Colab T4 ou Kaggle GPU).
- Dossier `data/` disponible dans la session (upload manuel ou clone du dépôt).

In [ ]:
%pip install -q unsloth trl datasets matplotlib

## 1. Configuration

In [ ]:
import json
import os
from pathlib import Path

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct'  # alternative : 'unsloth/Llama-3.2-3B-Instruct'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
SEED = 42
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
USE_WANDB = False  # True : journalise la perte dans Weights & Biases (WANDB_API_KEY requis)
HF_REPO_GGUF = None  # ex. 'user/invoice-extractor-gguf' pour pousser le GGUF sur le Hub
OUTPUT_DIR = Path('outputs')
RESULTS_DIR = Path('results')

SYSTEM_PROMPT = 'Tu es un extracteur de données. Réponds uniquement par un objet JSON valide.'
INSTRUCTION = (
    'Extrais les champs de la facture suivante au format JSON : supplier_name, '
    'invoice_date (AAAA-MM-JJ), total_ht, total_ttc, '
    'line_items (description, quantity, unit_price, total_ht).'
)

if USE_WANDB:
    os.environ['WANDB_PROJECT'] = 'invoice-extraction-qlora'

## 2. Chargement des données

In [ ]:
def find_data_dir():
    for candidate in [Path('data'), Path('/content/data'), Path('../data')]:
        if (candidate / 'train.jsonl').exists():
            return candidate
    raise FileNotFoundError('train.jsonl introuvable : uploader le dossier data/ dans la session.')


def load_jsonl(path):
    with path.open(encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]


DATA_DIR = find_data_dir()
train_rows = load_jsonl(DATA_DIR / 'train.jsonl')
val_rows = load_jsonl(DATA_DIR / 'val.jsonl')
print(f'train: {len(train_rows)} | val: {len(val_rows)}')

## 3. Modèle 4 bits et adaptateurs LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

## 4. Formatage ChatML

In [ ]:
from datasets import Dataset


def to_messages(row):
    user_content = f"{INSTRUCTION}\n\n{row['input']}"
    target = json.dumps(row['output'], ensure_ascii=False)
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': target},
    ]


def format_row(row):
    return {'text': tokenizer.apply_chat_template(to_messages(row), tokenize=False)}


train_ds = Dataset.from_list([format_row(r) for r in train_rows])
val_ds = Dataset.from_list([format_row(r) for r in val_rows])

token_lengths = [len(tokenizer(t)['input_ids']) for t in train_ds['text']]
print(f'longueur max en tokens : {max(token_lengths)} (limite : {MAX_SEQ_LENGTH})')
assert max(token_lengths) <= MAX_SEQ_LENGTH, 'des exemples dépassent MAX_SEQ_LENGTH'
print(train_ds[0]['text'][:600])

## 5. Entraînement avec SFTTrainer (TRL)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field='text',
    packing=False,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=5,
    lr_scheduler_type='linear',
    optim='adamw_8bit',
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_strategy='no',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    report_to='wandb' if USE_WANDB else 'none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats.metrics)

## 6. Journal et courbes de perte

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_points = [(h['step'], h['loss']) for h in history if 'loss' in h]
eval_points = [(h['step'], h['eval_loss']) for h in history if 'eval_loss' in h]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'train_log.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

plt.figure(figsize=(8, 4))
if train_points:
    plt.plot(*zip(*train_points), label='train loss')
if eval_points:
    plt.plot(*zip(*eval_points), label='val loss')
plt.xlabel('step')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.savefig(RESULTS_DIR / 'loss_curve.png', dpi=150)
plt.show()

In [ ]:
model.save_pretrained('lora_adapter')
tokenizer.save_pretrained('lora_adapter')

## 7. Vérification rapide

In [ ]:
FastLanguageModel.for_inference(model)

sample = val_rows[0]
prompt_messages = [m for m in to_messages(sample) if m['role'] != 'assistant']
inputs = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to(model.device)

outputs = model.generate(input_ids=inputs, max_new_tokens=512, do_sample=False, use_cache=True)
print('Prédit :', tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))
print('Attendu :', json.dumps(sample['output'], ensure_ascii=False))

## 8. Export GGUF Q4_K_M

In [ ]:
GGUF_DIR = 'gguf_q4_k_m'

model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method='q4_k_m')
print(list(Path(GGUF_DIR).glob('*.gguf')))

In [ ]:
if HF_REPO_GGUF:
    model.push_to_hub_gguf(
        HF_REPO_GGUF,
        tokenizer,
        quantization_method='q4_k_m',
        token=os.environ['HF_TOKEN'],
    )
else:
    print('HF_REPO_GGUF non défini : upload ignoré.')

## 9. Inférence avec llama.cpp

Exécuter ces commandes dans un terminal, à partir de la racine du projet, après avoir téléchargé le fichier `.gguf` :

```bash
git clone https://github.com/ggml-org/llama.cpp
cmake -B llama.cpp/build -S llama.cpp
cmake --build llama.cpp/build --config Release -j
./llama.cpp/build/bin/llama-cli -m gguf_q4_k_m/<fichier>.gguf -n 512 -p "Extrais les champs de la facture suivante au format JSON : ..."
```